# Figure S1: Benchmark Extension

- Panels: `FigS1a`, `FigS1b`, `FigS1c`, `FigS1d`
- Role: condition-level distribution extension for the main benchmark
- Inputs: `benchmark_summary_all.csv` + per-model `metrics.csv`
- Notes: Norman remains `single-only` to match the main benchmark definition


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / 'scripts').exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
if str(repo_root / 'src') not in sys.path:
    sys.path.insert(0, str(repo_root / 'src'))

from scripts.common.paper_plot_style import apply_gears_paper_style, style_axis as paper_style_axis
apply_gears_paper_style(font_scale=1.0)

DATASET_ORDER = ['Adamson', 'Norman', 'Dixit']
MODEL_ORDER = ['trishift_nearest', 'gears', 'biolord', 'genepert', 'scgpt']
MODEL_LABELS = {'trishift_nearest':'TriShift','gears':'GEARS','biolord':'biolord','genepert':'GenePert','scgpt':'scGPT'}
MODEL_COLORS = {'TriShift':'#9FD9D3','GEARS':'#F2B56B','biolord':'#F0806A','GenePert':'#87A8D8','scGPT':'#DDD3C8'}
OUT_ROOT = repo_root / 'artifacts' / 'paper_figures' / 'supp' / 'FigS1_BenchmarkExtension'
OUT_ROOT.mkdir(parents=True, exist_ok=True)

def _style_axis(ax):
    paper_style_axis(ax, grid_axis='y')
    for spine in ax.spines.values():
        spine.set_linewidth(0.8)



In [ ]:
summary_path = repo_root / 'artifacts' / 'paper_figures' / 'main' / 'Fig2_MultiDatasetBenchmark' / 'benchmark_summary_all.csv'
summary_df = pd.read_csv(summary_path)
summary_df = summary_df[summary_df['model_name'].isin(MODEL_ORDER)].copy()
summary_df['dataset_label'] = pd.Categorical(summary_df['dataset_label'], DATASET_ORDER, ordered=True)
summary_df['display_name'] = summary_df['model_name'].map(MODEL_LABELS)

detail_rows = []
for row in summary_df.itertuples(index=False):
    metrics_path = Path(row.metrics_path)
    if not metrics_path.exists():
        continue
    metrics_df = pd.read_csv(metrics_path)
    if str(row.dataset).strip().lower() == 'norman' and 'subgroup' in metrics_df.columns:
        metrics_df = metrics_df[metrics_df['subgroup'].astype(str).str.lower() == 'single'].copy()
    keep_cols = [c for c in ['condition','pearson','nmse','deg_mean_r2','systema_corr_20de_allpert'] if c in metrics_df.columns]
    if not keep_cols:
        continue
    keep = metrics_df[keep_cols].copy()
    keep['dataset'] = row.dataset
    keep['dataset_label'] = row.dataset_label
    keep['model_name'] = row.model_name
    keep['display_name'] = row.display_name
    detail_rows.append(keep)
detail_df = pd.concat(detail_rows, ignore_index=True) if detail_rows else pd.DataFrame()
detail_df['dataset_label'] = pd.Categorical(detail_df['dataset_label'], DATASET_ORDER, ordered=True)
display(detail_df.head())


In [ ]:
metric_specs = [
    ('pearson', 'Pearson', 'figs1a_pearson_boxplot.png'),
    ('nmse', 'nMSE', 'figs1b_nmse_boxplot.png'),
    ('deg_mean_r2', r'DEG mean $R^2$', 'figs1c_deg_mean_r2_boxplot.png'),
    ('systema_corr_20de_allpert', 'Systema Pearson', 'figs1d_systema_corr_boxplot.png'),
]

for metric_col, ylabel, out_name in metric_specs:
    plot_df = detail_df.dropna(subset=[metric_col]).copy()
    if plot_df.empty:
        continue
    fig, ax = plt.subplots(figsize=(7.2, 4.6), dpi=220)
    sns.boxplot(
        data=plot_df,
        x='dataset_label',
        y=metric_col,
        hue='display_name',
        order=DATASET_ORDER,
        hue_order=[MODEL_LABELS[m] for m in MODEL_ORDER],
        palette=MODEL_COLORS,
        linewidth=0.9,
        fliersize=1.8,
        width=0.72,
        ax=ax,
    )
    for patch in ax.artists:
        patch.set_edgecolor('black')
        patch.set_linewidth(0.8)
    for line in ax.lines:
        line.set_color('black')
        line.set_linewidth(0.8)
    ax.set_xlabel('')
    ax.set_ylabel(ylabel)
    ax.set_title(ylabel, pad=10)
    ax.legend(title='', frameon=False, ncol=3, loc='upper center', bbox_to_anchor=(0.5, 1.12), borderaxespad=0.0)
    _style_axis(ax)
    fig.tight_layout(rect=[0, 0, 1, 0.94])
    fig.savefig(OUT_ROOT / out_name, bbox_inches='tight')
    plot_df.to_csv(OUT_ROOT / out_name.replace('.png', '_values.csv'), index=False)
    plt.close(fig)
